In [1]:
import polars as pl
import importlib
import preprocessing
from datetime import datetime

/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
importlib.reload(preprocessing)
from preprocessing import NewsPreprocessing

In [ ]:
#Run the interactive pipeline
# files_to_load = [
#     "full_news/AAPL_news.csv",
#     "full_news/external_news.parquet"  
# ]

# # Initialize with the LIST of files
# preprocessor = NewsPreprocessing(files_to_load)

# Run the dialog (Now you can use Option 5 and Switch Sources)
#df_ready = preprocessor.run_interactive_pipeline()

Initializing NLTK resources...


In [4]:
#Create fake df for testing
from datetime import datetime

# Create a small fake DataFrame with 8 rows
fake_df = pl.DataFrame({
    "date_utc": [
        datetime(2020, 6, 1, 14, 30),   # Monday
        datetime(2020, 6, 1, 9, 15),    # Monday
        datetime(2020, 6, 1, 16, 45),   # Monday
        datetime(2020, 6, 5, 17, 20),   # Friday
        datetime(2020, 6, 6, 10, 0),    # Saturday (weekend)
        datetime(2020, 6, 7, 15, 30),   # Sunday (weekend)
        datetime(2020, 6, 8, 8, 45),    # Monday
        datetime(2020, 6, 9, 17, 10),   # Tuesday
    ],
    "Article_title": [
        "Tech stocks rally amid optimism",
        "Federal Reserve holds rates steady",
        "Oil prices drop on demand concerns",
        "Unemployment claims fall unexpectedly",
        "Markets closed for weekend trading",
        "Analysts predict strong earnings season",
        "New trade deal announced with Europe",
        "Inflation data exceeds expectations",
    ],
    "Stock_symbol": ["AAPL"] * 8  # Optional: if you need this column
})

# Add UTC timezone
fake_df = fake_df.with_columns(
    pl.col("date_utc").dt.replace_time_zone("UTC")
)

display(fake_df)

date_utc,Article_title,Stock_symbol
"datetime[μs, UTC]",str,str
2020-06-01 14:30:00 UTC,"""Tech stocks rally amid optimis…","""AAPL"""
2020-06-01 09:15:00 UTC,"""Federal Reserve holds rates st…","""AAPL"""
2020-06-01 16:45:00 UTC,"""Oil prices drop on demand conc…","""AAPL"""
2020-06-05 17:20:00 UTC,"""Unemployment claims fall unexp…","""AAPL"""
2020-06-06 10:00:00 UTC,"""Markets closed for weekend tra…","""AAPL"""
2020-06-07 15:30:00 UTC,"""Analysts predict strong earnin…","""AAPL"""
2020-06-08 08:45:00 UTC,"""New trade deal announced with …","""AAPL"""
2020-06-09 17:10:00 UTC,"""Inflation data exceeds expecta…","""AAPL"""


In [3]:
source_file = "/mnt/windows/windows_hanka_bcthesis/full_news/nasdaq_external_news.parquet"
#source_file = "/mnt/windows/windows_hanka_bcthesis/full_news/AAPL_news.csv"
preprocessor = NewsPreprocessing(source_file)                                            
#loaded_df = preprocessor.load_data(source_file, filter_russian=True)
#with pl.Config(fmt_str_lengths=200, tbl_width_chars=1000, tbl_rows=20):
#   display(loaded_df.sort("date_est"))


Initializing NLTK resources...


In [5]:
loaded_df.sort("date_utc")

date_est,date_utc,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
"datetime[μs, America/New_York]","datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str
1914-09-16 19:00:00 EST,1914-09-17 00:00:00 UTC,"""1914. Das ist Nesteroff!""",null,"""https://lenta.ru/news/1914/09/…","""Первая мировая""","""Библиотека""","""Штабс-капитан П. Н. Нестеров н…",null,null,null,null
1969-12-30 19:00:00 EST,1969-12-31 00:00:00 UTC,"""Montpelier Re Holdings Ltd. (M…","""MRH""","""http://www.zacks.com/stock/res…","""Zacks""",null,null,null,null,null,null
2006-10-19 20:00:00 EDT,2006-10-20 00:00:00 UTC,"""Inco's Net Soars on Higher Met…",null,"""http://www.bloomberg.com/news/…",null,"""Dale Crofts""","""Inco Ltd., the Canadian nickel…",null,null,null,null
2006-10-20 20:00:00 EDT,2006-10-21 00:00:00 UTC,"""Jim Cramer: Diageo, Anheuser-B…",null,"""http://www.bloomberg.com/news/…",null,"""Steven Bodzin""","""Jim Cramer recommended that v…",null,null,null,null
2006-10-22 20:00:00 EDT,2006-10-23 00:00:00 UTC,"""Ex-Plant Worker Shuster Pleads…",null,"""http://www.bloomberg.com/news/…",null,"""David Glovin""","""A former worker at a Wisconsi…",null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
2020-06-11 08:49:41 EDT,2020-06-11 12:49:41 UTC,"""7 Stocks Moving In Thursday's …","""PVH""","""https://www.benzinga.com/news/…","""Tyree Gorges""",null,null,null,null,null,null
2020-06-11 08:51:33 EDT,2020-06-11 12:51:33 UTC,"""Financials, Energy Among Worst…","""WMT""","""https://www.benzinga.com/news/…","""JJ Kinahan""",null,null,null,null,null,null
2020-06-11 09:01:39 EDT,2020-06-11 13:01:39 UTC,"""Twitter Removes About 174,000 …","""TWTR""","""https://www.benzinga.com/tech/…","""Benzinga Newsdesk""",null,null,null,null,null,null


In [4]:
output_path = "/mnt/windows/windows_hanka_bcthesis/full_news/grouped_news_nasdaq.parquet"
#output_path = "/mnt/windows/windows_hanka_bcthesis/full_news/grouped_news_AAPL.parquet"
grouped_df = preprocessor.group_articles_by_trading_day(output_path = output_path, target_df = loaded_df)


 Grouping articles by trading session (cutoff: 20:00 UTC)...
Executing lazy join and grouping operations via Stream Engine to conserve RAM...
Returning LazyFrame.
Streaming complete. Data saved to /mnt/windows/windows_hanka_bcthesis/full_news/grouped_news_nasdaq.parquet


In [ ]:
# input_path = "/mnt/windows/windows_hanka_bcthesis/full_news/grouped_news_nasdaq.parquet"
# output_path = "/mnt/windows/windows_hanka_bcthesis/full_news/cleaned_news_nasdaq.parquet"
cleaned_df = preprocessor.clean_text(input_path = input_path, output_path = output_path)


 Starting Chunker Advanced Text Cleaning (Regex + Stemming)...
 Reading from: /mnt/windows/windows_hanka_bcthesis/full_news/grouped_news_nasdaq.parquet
 Saving to: /mnt/windows/windows_hanka_bcthesis/full_news/cleaned_news_nasdaq.parquet
 Total rows to process: 4328. Processing in batches of 50000...


NLTK Cleaning Chunks:   0%|          | 0/4328 [00:00<?, ?it/s]

NLTK Cleaning Chunks:   0%|          | 0/4328 [14:07<?, ?it/s]


KeyboardInterrupt: 

In [9]:
input_path = "/mnt/windows/windows_hanka_bcthesis/full_news/AAPL_news_cleaned.parquet"
test =pl.read_parquet(input_path)
test

trading_session_date_utc,daily_text,preprocessed_for_tfidf
date,str,str
2020-06-01,"""Apple Cuts iPhone Prices in Ch…","""appl cut iphon price china pus…"
2020-06-02,"""'Apple is tracking iPhones sto…","""appl track iphon stolen looter…"
2020-06-09,"""Why Apple's Stock Is Trading H…","""appl stock trade higher today …"
2020-06-10,"""Tech Stocks And FAANGS Strong …","""tech stock faang strong start …"
